In [1]:
# Use %pip (not !pip) so packages land in THIS kernel's environment.
# RAG also needs torch (model), and datasets + faiss (the retriever index).
# datasets must stay <4 because wiki_dpr is a script-based dataset.
%pip install -q transformers torch "datasets<4" faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# wiki_dpr is a script-based dataset, so datasets needs permission to run it.
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"
# torch and faiss each bundle their own libomp. On macOS that combination
# aborts with "OMP: Error #15"; KMP_DUPLICATE_LIB_OK downgrades the abort, and
# importing faiss FIRST avoids the segfault that otherwise follows.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import faiss  # noqa: F401  -- must be imported before torch/transformers

from transformers import RagTokenizer, RagRetriever, RagTokenForGeneration, RagSequenceForGeneration

tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
# The RAG config still points at the old single-segment id "wiki_dpr", which
# huggingface_hub 1.x rejects. Override it with the namespaced repo id.
retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="exact",
    use_dummy_dataset=True,
    dataset="facebook/wiki_dpr",
)
model = RagTokenForGeneration.from_pretrained("facebook/rag-token-nq", retriever=retriever)
question = "Who is the president of France?"
input_ids = tokenizer(question, return_tensors="pt").input_ids
output = model.generate(input_ids)
answer = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(answer)


/Users/ankit/.local/share/uv/python/cpython-3.13.14-macos-aarch64-none/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/ankit/.local/share/uv/python/cpython-3.13.14-macos-aarch64-none/lib/python3.13/site-packages/datasets/load.py:1231: FutureWarning: The repository for facebook/wiki_dpr contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/wiki_dpr
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
docs = retriever(input_ids.numpy(), return_tensors="pt")
doc_titles = tokenizer.batch_decode(docs["doc_ids"], skip_special_tokens=True)
print(doc_titles)


In [ ]:
probs = model(input_ids, labels=output, return_dict=True).logits
probs = probs.softmax(dim=-1)
top_tokens = tokenizer.batch_decode(probs[0, -1].topk(5).indices)
print(top_tokens)
